# Breast Cancer Detection Using Machine Learning
Educational project using the Wisconsin Diagnostic Breast Cancer dataset.

**Disclaimer:** Not for medical diagnosis or treatment. This model uses 30 numerical features, not images.

## 1. Load the dataset
The repository includes `data/data.csv`. In Colab, upload that CSV when prompted, or mount the repository files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    df = pd.read_csv('../data/data.csv')
except Exception:
    from google.colab import files
    uploaded = files.upload()  # upload data.csv
    csv_file = next(name for name in uploaded if name.lower().endswith('.csv'))
    df = pd.read_csv(csv_file)

print('Shape:', df.shape)
display(df.head())
df.info()

## 2. Clean and inspect

In [ ]:
print('Missing values before cleaning:', df.isna().sum().sum())
print('Duplicate rows before cleaning:', df.duplicated().sum())
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')
df['diagnosis'] = df['diagnosis'].map({'B': 0, 'M': 1})
if df['diagnosis'].isna().any():
    raise ValueError('Unexpected diagnosis labels')
print('Shape after cleaning:', df.shape)
print('Class counts (0=Benign, 1=Malignant):')
print(df['diagnosis'].value_counts())
display(df.describe())

## 3. Exploratory data analysis

In [ ]:
sns.countplot(data=df, x='diagnosis')
plt.title('Diagnosis distribution (0=Benign, 1=Malignant)')
plt.show()

numeric_features = df.drop(columns='diagnosis')
plt.figure(figsize=(16, 12))
sns.heatmap(numeric_features.corr(), cmap='coolwarm', center=0)
plt.title('Feature correlation heatmap')
plt.show()

features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean']
df[features].hist(figsize=(14, 9), bins=20)
plt.tight_layout(); plt.show()

## 4. Split and scale without data leakage
The scaler is fitted only on the training split. The test set is kept aside for final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
X = df.drop(columns='diagnosis')
y = df['diagnosis'].astype(int)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)
print(X_train.shape, X_test.shape)

## 5. Train and compare six classifiers

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

models = {
 'Logistic Regression': LogisticRegression(max_iter=3000),
 'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
 'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
 'SVM': SVC(kernel='rbf', class_weight='balanced'),
 'KNN': KNeighborsClassifier(n_neighbors=5),
 'XGBoost': XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, eval_metric='logloss')
}
trained_models = {}
rows = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    p = clf.predict(X_test)
    trained_models[name] = clf
    rows.append({'Model': name, 'Accuracy': accuracy_score(y_test,p), 'Precision_Malignant': precision_score(y_test,p,zero_division=0), 'Recall_Malignant': recall_score(y_test,p,zero_division=0), 'F1_Malignant': f1_score(y_test,p,zero_division=0)})
results_df = pd.DataFrame(rows).sort_values('Recall_Malignant', ascending=False)
display(results_df)

## 6. Hyperparameter tuning with stratified 5-fold cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_spaces = {
 'Logistic Regression': (LogisticRegression(max_iter=3000), {'C':[0.01,0.1,1,10], 'solver':['liblinear','lbfgs']}),
 'Decision Tree': (DecisionTreeClassifier(random_state=42, class_weight='balanced'), {'max_depth':[None,5,10,15], 'min_samples_split':[2,5,10], 'min_samples_leaf':[1,2,4]}),
 'Random Forest': (RandomForestClassifier(random_state=42, class_weight='balanced'), {'n_estimators':[100,200], 'max_depth':[None,5,10], 'min_samples_split':[2,5]}),
 'SVM': (SVC(class_weight='balanced'), {'C':[0.1,1,10], 'kernel':['linear','rbf'], 'gamma':['scale','auto']}),
 'KNN': (KNeighborsClassifier(), {'n_neighbors':[3,5,7,9], 'weights':['uniform','distance'], 'p':[1,2]}),
 'XGBoost': (XGBClassifier(random_state=42, eval_metric='logloss'), {'n_estimators':[100,200], 'max_depth':[2,3,5], 'learning_rate':[0.01,0.05,0.1]})
}
tuned_models = {}
tune_rows = []
for name, (estimator, grid) in search_spaces.items():
    search = GridSearchCV(estimator, grid, scoring='recall', cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    p = search.best_estimator_.predict(X_test)
    tune_rows.append({'Model':name, 'CV_Recall':search.best_score_, 'Test_Accuracy':accuracy_score(y_test,p), 'Test_Precision':precision_score(y_test,p,zero_division=0), 'Test_Recall':recall_score(y_test,p,zero_division=0), 'Test_F1':f1_score(y_test,p,zero_division=0), 'Best_Params':search.best_params_})
tuning_df = pd.DataFrame(tune_rows).sort_values('CV_Recall', ascending=False)
display(tuning_df)

## 7. Confusion matrix and detailed report

In [ ]:
for name, clf in tuned_models.items():
    p = clf.predict(X_test)
    print('\n', name)
    print(classification_report(y_test, p, target_names=['Benign','Malignant'], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, p, display_labels=['Benign','Malignant'], cmap='Blues')
    plt.title(name)
    plt.show()

## 8. SHAP explainability for tuned Logistic Regression

In [ ]:
!pip -q install shap
import shap
best_model = tuned_models['Logistic Regression']
explainer = shap.LinearExplainer(best_model, X_train)
shap_values = explainer(X_test)
shap_values.feature_names = X.columns.tolist()
shap.plots.beeswarm(shap_values, max_display=15)
shap_importance = pd.Series(np.abs(shap_values.values).mean(axis=0), index=X.columns).sort_values(ascending=False)
display(shap_importance.head(15))

## 9. Save model artifacts for Streamlit

In [ ]:
import joblib, json, os
from pathlib import Path
os.makedirs('models', exist_ok=True)
final_model = tuned_models['Logistic Regression']
joblib.dump(final_model, 'models/breast_cancer_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
Path('models/feature_names.json').write_text(json.dumps(X.columns.tolist(), indent=2))
Path('models/feature_medians.json').write_text(json.dumps({k:float(v) for k,v in X_train_raw.median().to_dict().items()}, indent=2))
print('Artifacts saved in models/')

## 10. Test one held-out sample

In [ ]:
sample = X_test[0:1]
pred = int(final_model.predict(sample)[0])
proba = final_model.predict_proba(sample)[0]
print('Predicted class:', 'Malignant' if pred == 1 else 'Benign')
print('Class probabilities:', dict(zip(final_model.classes_.tolist(), proba.tolist())))